In [ ]:
!pip install darts

In [ ]:
import sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from google.colab import files
from pathlib import Path
import warnings
import lightgbm as lgb
import seaborn as sns
import darts
from darts.models import LightGBMModel, ExponentialSmoothing
from darts import TimeSeries
import concurrent.futures
from sklearn.metrics import mean_squared_log_error
from darts.utils.utils import SeasonalityMode

In [ ]:
warnings.filterwarnings('ignore')

In [ ]:
def rmsle(actual, predictions):
    return np.sqrt(mean_squared_log_error(actual, predictions))

In [ ]:
comp_dir = Path('../input/store-sales-time-series-forecasting')

train = pd.read_csv(
    comp_dir / 'train.csv',
    usecols=['store_nbr', 'family', 'date', 'sales', 'onpromotion'],
    dtype={
        'store_nbr': 'category',
        'family': 'category',
        'sales': 'float32',
        'onpromotion': 'uint32',
    },
    parse_dates=['date'],
    infer_datetime_format=True,
)

train.sort_values(by=['family', 'store_nbr', 'date'], inplace=True)

train = train.set_index('date')

In [ ]:
test = pd.read_csv(
    comp_dir / 'test.csv',
    dtype={
        'store_nbr': 'category',
        'family': 'category',
        'onpromotion': 'uint32',
    },
    parse_dates=['date'],
    infer_datetime_format=True,
)

test = test.set_index('date')

In [ ]:
train_families = {family: train[train['family'] == family] for family in train['family'].unique()}

for train_family in train_families:
    train_families[train_family] = train_families[train_family].drop(columns=['family', 'onpromotion'])
    train_families[train_family] = train_families[train_family].pivot(columns='store_nbr', values='sales')

In [ ]:
test_families = {family: test[test['family'] == family] for family in test['family'].unique()}

# for test_family in test_families:
#     test_families[test_family] = test_families[test_family].pivot(columns='store_nbr')

In [ ]:
def create_features(data):
  features = pd.DataFrame()
  features['date'] = pd.to_datetime(data['Date'])

  features['Day of the Week'] = features['date'].dt.weekday
  features['Month'] = features['date'].dt.month
  features['Day of the Month'] = features['date'].dt.day
  features['Day of the Year'] = features['date'].dt.dayofyear
  features['Year'] = features['date'].dt.year
  features['New Years'] = features['date'].dt.dayofyear == 1
  features['New Years Eve'] = (features['date'].dt.day == 31) & (features['date'].dt.month == 12)
  features['Christmas'] = (features['date'].dt.day == 25) & (features['date'].dt.month == 12)

  features['Store Number'] = data['Store Number']
  features = pd.get_dummies(features, columns=['New Years', 'New Years Eve', 'Christmas'])
    
  return features

In [ ]:
def XGB_Model(train, test):
  train_features = create_features(train)
  test_features = create_features(test)

  train_features_without_date = train_features.copy().drop(columns='date')
  test_features_without_date = test_features.copy().drop(columns='date')

  model = LightGBMModel()
  model = model.fit(train['Predicted Sales'], future_covariates = TimeSeries.from_dataframe(train_features_without_date))

  predictions = pd.DataFrame()
  predictions['Predicted Sales'] = model.predict(test_features_without_date)
  predictions['Store Number'] = test_features['Store Number']
  predictions['date'] = test_features['date']

  fitted_values = pd.DataFrame()
  fitted_values['Predicted Sales'] = model.predict(train_features_without_date)
  fitted_values['Store Number'] = train_features['Store Number']
  fitted_values['date'] = train_features['date']

  return predictions, fitted_values

In [ ]:
def plot(residuals):
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x=residuals.index, y=residuals['Predicted Sales'])
    
    # Adding a horizontal line at 0 (indicating perfect predictions)
    plt.axhline(0, color='red', linestyle='--')
    
    # Labels and title
    plt.xlabel('Predicted Sales')
    plt.ylabel('Residuals')
    plt.title('Residual Plot')
    
    plt.show()

In [ ]:
from joblib import Parallel, delayed

# def process_column(train, test, column):
#     train = train.loc[:, column]
#     p, r = es_model(train, test, column)
#     return p, r, train[-170:][:160]

def process_column(train, test, column):
    train = train.loc[:, column]
    train = np.log(train + 1)
    p = es_model(train, test, column)
    return p

In [ ]:
def es_model(train, test, store_nbr):
    # index = train.index.unique()[-170:][:160]
    train = TimeSeries.from_series(train, freq='D')
    
    model = ExponentialSmoothing(seasonal=SeasonalityMode.ADDITIVE, seasonal_periods=7)
    model = model.fit(train)

    predictions = pd.DataFrame({
        'Predicted Sales': pd.Series(model.predict(n=16).values().flatten()),
        'Store Number': store_nbr,
        'Date': test.index.unique()
    })

    # residuals = model.residuals(
    #         train, 
    #         stride=16,
    #         last_points_only=False,
    #         retrain=True,
    #         forecast_horizon=16,
    #         # verbose=True,
    #         start=0.9
    # )
    
    # r = pd.Series()
    # for each in residuals:
    #     r = pd.concat([r, each.pd_series()])

    # residuals = pd.DataFrame({
    #     'Predicted Sales': r,
    #     'Store Number': store_nbr,
    #     'Date': index
    # })
    
    # return predictions, residuals

    return predictions

In [ ]:
def family_predictions(train, test):
    es_predictions = pd.DataFrame()
    es_residuals = pd.DataFrame()
    
    train = train.fillna(0)
    train = train.reindex(pd.date_range(start="2013-01-01", end="2017-08-15", freq="D"), fill_value = 0)
    
    results = Parallel(n_jobs=-1)(delayed(process_column)(train, test, column) for column in train.columns)
    es_predictions = pd.concat([res for res in results])
    es_predictions['Predicted Sales'] = es_predictions['Predicted Sales'].clip(lower=0)


    es_predictions['Predicted Sales'] = np.exp(es_predictions['Predicted Sales']) - 1



    # results = Parallel(n_jobs=-1)(delayed(process_column)(train, test, column) for column in train.columns)
    # es_predictions = pd.concat([res[0] for res in results])
    # es_residuals = pd.concat([res[1] for res in results])
    # train = pd.concat([res[2] for res in results])

    # historical_forecasts = (train - es_residuals['Predicted Sales']).clip(lower=0)
    
    # error = rmsle(train, historical_forecasts)
    # print(error)

    # plot(es_residuals)

    
 
    # r = es_residuals.copy()
    # es_residuals = es_residuals.reset_index()
    # es_predictions = es_predictions.reset_index()

    # XGB_Predictions, XGB_fit_values = XGB_Model(es_residuals, es_predictions)
    # x = XGB_fit_values.copy()
    # x.index = r.index
    # print(x.index)
    # plot(r + x)
    # XGB_Predictions['Predicted Sales'] += es_predictions['Predicted Sales']
            
    # return XGB_Predictions
    return es_predictions

In [ ]:
predictions = pd.DataFrame()
scores = pd.Series()
for train_family in train_families:
    print(train_family)
    induvidual_predictions = family_predictions(train_families[train_family], test_families[train_family])
    # induvidual_predictions, score = family_predictions(train_families[train_family], test_families[train_family])
    induvidual_predictions['Family'] = train_family
    predictions = pd.concat([predictions, induvidual_predictions])
    # scores.loc[len(scores)] = score
    
print(predictions)

In [ ]:
adsaiooaj = predictions.copy()
dasfsd = test.copy()

In [ ]:
predictions = adsaiooaj.copy()
predictions['family'] = predictions['Family']
predictions['store_nbr'] = predictions['Store Number'].astype(str)
predictions['sales'] = predictions['Predicted Sales']
predictions['date'] = predictions['Date']
predictions = predictions.reset_index()
predictions = predictions.drop(columns=['Family', 'Store Number', 'index', 'Predicted Sales', 'Date'])
predictions = predictions.set_index(['family', 'store_nbr', 'date'])


In [ ]:
test = dasfsd.reset_index()
test = test.set_index(['family', 'store_nbr', 'date'])
test = test.drop(columns='onpromotion')

In [ ]:
predictions

In [ ]:
test

In [ ]:
submission = predictions.join(test.id).reindex(columns=['id', 'sales'])
submission.to_csv('submission.csv', index=False)
submission

In [ ]:
print(len(submission))